In [1]:

import time
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

from sklearn.model_selection import (
    TimeSeriesSplit, GridSearchCV, RandomizedSearchCV
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report
)

RANDOM_STATE = 42

sns.set_style("whitegrid")
plt.rcParams.update({
    "figure.dpi": 100,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
})

# ==========================================================================
# HYPERPARAMETER GRIDS
# ==========================================================================

LR_GRID = {
    "model__C": [0.001, 0.01, 0.1, 1, 10, 100]
}

SVM_GRID = {
    "model__kernel": ["rbf"],
    "model__C": [0.1, 1, 10, 100],
    "model__gamma": ["scale", "auto", 0.01, 0.1]
}

RF_GRID = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [None, 10, 20, 30],
    "model__min_samples_split": [2, 5, 10]
}

# ==========================================================================
# HELPER FUNCTIONS
# ==========================================================================

def create_temporal_cv(periods, n_splits):


    tscv = TimeSeriesSplit(n_splits=n_splits)

    cv_indices = []

    for train_period_idx, val_period_idx in tscv.split(periods):

        train_periods = periods.iloc[train_period_idx]
        val_periods = periods.iloc[val_period_idx]

        train_keys = pd.MultiIndex.from_frame(train_periods)
        val_keys = pd.MultiIndex.from_frame(val_periods)

        row_keys = pd.MultiIndex.from_frame(periods.attrs["row_periods"])

        train_mask = row_keys.isin(train_keys)
        val_mask = row_keys.isin(val_keys)

        train_rows = np.flatnonzero(train_mask)
        val_rows = np.flatnonzero(val_mask)

        cv_indices.append((train_rows, val_rows))

    return cv_indices


def make_model(model):

    return Pipeline([
        ("scaler", StandardScaler()),
        ("model", model)
    ])


# ==========================================================================
# LOAD DATA
# ==========================================================================

df_raw = pd.read_csv("Processed Data.csv", parse_dates=["date"])

df_raw["grid_cell_id"] = (
    df_raw["Latitude"].astype(str) + "_" +
    df_raw["Longitude"].astype(str)
)

##############################################################################
# TASK 1: SEASONAL RAINFALL MODELLING
##############################################################################

print("=" * 140)
print("TASK 1: SEASONAL RAINFALL -- CHRONOLOGICAL")
print("=" * 140)

DISTRICT_ORDER = [
    "Burera", "Gakenke", "Musanze", "Ngororero",
    "Nyabihu", "Rubavu", "Rutsiro"
]

SEASON_ORDER = [
    "Season A (Rainy)", "Season B (Rainy)", "Dry Season"
]

CATEGORY_ORDER = [
    "Below Normal", "Normal", "Above Normal"
]

season_rank = {
    season: i for i, season in enumerate(SEASON_ORDER)
}

# --------------------------------------------------------------------------
# Seasonal aggregation
# --------------------------------------------------------------------------

grid_seasonal = (
    df_raw.groupby(
        ["District", "grid_cell_id", "Year", "Season"]
    )
    .agg(
        pr_seasonal_total=("pr", "sum"),
        tas_seasonal_mean=("tas", "mean"),
        tasmin_seasonal_mean=("tasmin", "mean"),
        tasmax_seasonal_mean=("tasmax", "mean"),
        temp_range_seasonal_mean=("temp_range", "mean")
    )
    .reset_index()
)

seasonal = (
    grid_seasonal.groupby(["District", "Year", "Season"])
    .agg(
        pr_seasonal_total=("pr_seasonal_total", "mean"),
        tas_seasonal_mean=("tas_seasonal_mean", "mean"),
        tasmin_seasonal_mean=("tasmin_seasonal_mean", "mean"),
        tasmax_seasonal_mean=("tasmax_seasonal_mean", "mean"),
        temp_range_seasonal_mean=("temp_range_seasonal_mean", "mean")
    )
    .reset_index()
)

seasonal["_season_order"] = seasonal["Season"].map(season_rank)

seasonal = seasonal.sort_values(
    ["District", "Year", "_season_order"]
).reset_index(drop=True)

# --------------------------------------------------------------------------
# Rainfall category identification
# --------------------------------------------------------------------------

seasonal["Rainfall_Category"] = None

for (district, season), group in seasonal.groupby(
    ["District", "Season"]
):
    p33 = group["pr_seasonal_total"].quantile(1 / 3)
    p67 = group["pr_seasonal_total"].quantile(2 / 3)

    cat = pd.cut(
        group["pr_seasonal_total"],
        bins=[-np.inf, p33, p67, np.inf],
        labels=CATEGORY_ORDER
    )

    seasonal.loc[group.index, "Rainfall_Category"] = cat

# --------------------------------------------------------------------------
# Laghed feature engineering
# --------------------------------------------------------------------------

lag_src = [
    "pr_seasonal_total",
    "tas_seasonal_mean",
    "tasmin_seasonal_mean",
    "tasmax_seasonal_mean",
    "temp_range_seasonal_mean"
]

lag_new = [
    "prior year rainfall mm",
    "prior year mean tas",
    "prior year mean tasmin",
    "prior year mean tasmax",
    "prior year temp range"
]

seasonal = seasonal.sort_values(
    ["District", "Season", "Year"]
)

for src, new in zip(lag_src, lag_new):
    seasonal[new] = (
        seasonal.groupby(["District", "Season"])[src].shift(1)
    )

model_df = seasonal.dropna(subset=lag_new).copy()

# --------------------------------------------------------------------------
# Categorical Variables encoding
# --------------------------------------------------------------------------

district_dummies = pd.get_dummies(
    model_df["District"], prefix="District", dtype=int
)

season_dummies = pd.get_dummies(
    model_df["Season"], prefix="Season", dtype=int
)

model_df = pd.concat(
    [model_df, district_dummies, season_dummies],
    axis=1
)

FEATURE_COLS_A = (
    lag_new + ["Year"] +
    list(district_dummies.columns) +
    list(season_dummies.columns)
)

# --------------------------------------------------------------------------
# CHRONOLOGICAL--TRAIN-TEST SPLIT
# --------------------------------------------------------------------------

model_df = model_df.sort_values(
    ["Year", "_season_order", "District"]
).reset_index(drop=True)

periods_a = (
    model_df[["Year", "Season"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

split_index_a = int(len(periods_a) * 0.80)

train_periods_a = periods_a.iloc[:split_index_a].copy()
test_periods_a = periods_a.iloc[split_index_a:].copy()

train_keys_a = pd.MultiIndex.from_frame(train_periods_a)
test_keys_a = pd.MultiIndex.from_frame(test_periods_a)

row_keys_a = pd.MultiIndex.from_frame(
    model_df[["Year", "Season"]]
)

train_mask_a = row_keys_a.isin(train_keys_a)
test_mask_a = row_keys_a.isin(test_keys_a)

X_a = model_df[FEATURE_COLS_A].copy()
y_a = model_df["Rainfall_Category"].astype(str).copy()

X_train_a = X_a.loc[train_mask_a].reset_index(drop=True)
X_test_a = X_a.loc[test_mask_a].reset_index(drop=True)

y_train_a = y_a.loc[train_mask_a].reset_index(drop=True)
y_test_a = y_a.loc[test_mask_a].reset_index(drop=True)

print(f"Training rows: {len(X_train_a)}")
print(f"Testing rows: {len(X_test_a)}")

print(
    f"Training period: {train_periods_a.iloc[0]['Year']} - "
    f"{train_periods_a.iloc[-1]['Year']}"
)

print(
    f"Testing period: {test_periods_a.iloc[0]['Year']} - "
    f"{test_periods_a.iloc[-1]['Year']}"
)

# --------------------------------------------------------------------------
# TIME SERIES CROSS-VALIDATION For TASK 1
# --------------------------------------------------------------------------

row_periods_train_a = pd.MultiIndex.from_frame(
    model_df.loc[train_mask_a, ["Year", "Season"]].reset_index(drop=True)
)

cv_periods_a = train_periods_a.reset_index(drop=True)

tscv_a = TimeSeriesSplit(n_splits=5)

cv_a = []

for train_idx, val_idx in tscv_a.split(cv_periods_a):

    train_keys = pd.MultiIndex.from_frame(
        cv_periods_a.iloc[train_idx]
    )

    val_keys = pd.MultiIndex.from_frame(
        cv_periods_a.iloc[val_idx]
    )

    train_rows = np.flatnonzero(
        row_periods_train_a.isin(train_keys)
    )

    val_rows = np.flatnonzero(
        row_periods_train_a.isin(val_keys)
    )

    cv_a.append((train_rows, val_rows))

print(f"Task 1 TimeSeriesSplit folds: {len(cv_a)}")

# --------------------------------------------------------------------------
# MODEL DEFINITIONS For TASK 1
# --------------------------------------------------------------------------

baseline_models_a = {
    "Logistic Regression": make_model(
        LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
    ),

    "Random Forest": make_model(
        RandomForestClassifier(
            n_estimators=300,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    ),

    "SVM (RBF Kernel)": make_model(
        SVC(
            kernel="rbf",
            C=1.0,
            gamma="scale",
            random_state=RANDOM_STATE
        )
    )
}

search_configs_a = {
    "Logistic Regression": (
        make_model(
            LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
        ), LR_GRID
    ),

    "Random Forest": (
        make_model(
            RandomForestClassifier(
                random_state=RANDOM_STATE,
                n_jobs=-1
            )
        ), RF_GRID
    ),

    "SVM (RBF Kernel)": (
        make_model(
            SVC(random_state=RANDOM_STATE)
        ), SVM_GRID
    )
}

# --------------------------------------------------------------------------
# HYPERPARAMETER TUNING for TASK 1
# --------------------------------------------------------------------------

best_params_a = {}
tuned_models_a = {}

for name, (estimator, grid) in search_configs_a.items():

    t0 = time.time()

    search = GridSearchCV(
        estimator,
        grid,
        cv=cv_a,
        scoring="f1_macro",
        n_jobs=1
    )

    search.fit(X_train_a, y_train_a)

    elapsed = time.time() - t0

    best_params_a[name] = search.best_params_
    tuned_models_a[name] = search.best_estimator_

    print(
        f"-> {name}: best temporal CV f1_macro = "
        f"{search.best_score_:.4f} | "
        f"best params = {search.best_params_} | "
        f"{elapsed:.1f}s"
    )

rows = [
    {"Model": name, "Best Hyperparameters": str(params)}
    for name, params in best_params_a.items()
]

pd.DataFrame(rows).to_csv(
    "Seasonal Best Hyperparameters.csv", index=False
)

# --------------------------------------------------------------------------
# BASELINE VS TUNED PERFORMANCE - TASK 1
# --------------------------------------------------------------------------

perf_rows = []

for name in baseline_models_a:

    baseline_models_a[name].fit(X_train_a, y_train_a)

    y_pred_base = baseline_models_a[name].predict(X_test_a)
    y_pred_tuned = tuned_models_a[name].predict(X_test_a)

    for label, y_pred in [
        ("Baseline", y_pred_base),
        ("Tuned", y_pred_tuned)
    ]:

        perf_rows.append({
            "Model": name,
            "Version": label,
            "Accuracy": round(
                accuracy_score(y_test_a, y_pred), 4
            ),
            "F1-Score (macro)": round(
                f1_score(
                    y_test_a, y_pred,
                    average="macro",
                    zero_division=0
                ), 4
            )
        })

seasonal_comparison = pd.DataFrame(perf_rows)

print("\n" + seasonal_comparison.to_string(index=False))

seasonal_comparison.to_csv(
    "Seasonal Tuned vs Baseline.csv", index=False
)

fig, ax = plt.subplots(figsize=(9, 5.5))

pivot = seasonal_comparison.pivot(
    index="Model",
    columns="Version",
    values="F1-Score (macro)"
)

pivot[["Baseline", "Tuned"]].plot(
    kind="bar", ax=ax,
    color=["#B0BEC5", "#2E86AB"]
)

ax.set_title(
    "Task 1: Seasonal Rainfall Category — Baseline vs Tuned"
)
ax.set_ylabel("F1-Score — macro")
ax.set_ylim(0, 1)

plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("Seasonal Tuned vs Baseline.png")
plt.close()

print("\nSaved: Seasonal Tuned vs Baseline.png")

# --------------------------------------------------------------------------
#  CLASSIFICATION REPORT-SEASONAL
# --------------------------------------------------------------------------

report_rows = []

for name, model in tuned_models_a.items():

    y_pred = model.predict(X_test_a)

    report = classification_report(
        y_test_a,
        y_pred,
        output_dict=True,
        zero_division=0
    )

    for category in CATEGORY_ORDER:

        if category in report:

            r = report[category]

            report_rows.append({
                "Model": name,
                "Category": category,
                "Precision": round(r["precision"], 3),
                "Recall": round(r["recall"], 3),
                "F1-Score": round(r["f1-score"], 3),
                "Support": int(r["support"])
            })

pd.DataFrame(report_rows).to_csv(
    "Seasonal Tuned Classification Report.csv",
    index=False
)


##############################################################################
# TASK 2: EXTREME RAINFALL MODELLING
##############################################################################

print("\n" + "=" * 140)
print("TASK 2: EXTREME RAINFALL -- CHRONOLOGICAL")
print("=" * 140)

# --------------------------------------------------------------------------
# Sorting observations and creating lagged variables
# --------------------------------------------------------------------------

df_b = df_raw.sort_values(
    ["grid_cell_id", "Year", "Month"]
).reset_index(drop=True)

for src, new in zip(
    ["tas", "tasmin", "tasmax", "temp_range"],
    ["tas_lag1", "tasmin_lag1", "tasmax_lag1", "temp_range_lag1"]
):

    df_b[new] = df_b.groupby("grid_cell_id")[src].shift(1)

    df_b[new] = df_b.groupby("grid_cell_id")[new].transform(
        lambda s: s.fillna(s.mean())
    )

season_dummies_b = pd.get_dummies(
    df_b["Season"], prefix="Season", dtype=int
)

df_b = pd.concat(
    [df_b, season_dummies_b],
    axis=1
)

district_cols_b = [
    c for c in df_b.columns
    if c.startswith("District_")
]

FEATURE_COLS_B = (
    [
        "pr_lag1", "pr_lag2", "pr_lag3",
        "tas_lag1", "tasmin_lag1",
        "tasmax_lag1", "temp_range_lag1"
    ]
    + district_cols_b
    + list(season_dummies_b.columns)
    + ["Year"]
)

# --------------------------------------------------------------------------
# CHRONOLOGICAL--TRAIN-TEST SPLIT
# --------------------------------------------------------------------------

df_b = df_b.sort_values(
    ["Year", "Month", "grid_cell_id"]
).reset_index(drop=True)

X_b = df_b[FEATURE_COLS_B].copy()
y_b = df_b["is_extreme_rainfall"].copy()

periods_b = (
    df_b[["Year", "Month"]]
    .drop_duplicates()
    .sort_values(["Year", "Month"])
    .reset_index(drop=True)
)

split_index_b = int(len(periods_b) * 0.80)

train_periods_b = periods_b.iloc[:split_index_b].copy()
test_periods_b = periods_b.iloc[split_index_b:].copy()

train_keys_b = pd.MultiIndex.from_frame(train_periods_b)
test_keys_b = pd.MultiIndex.from_frame(test_periods_b)

row_keys_b = pd.MultiIndex.from_frame(
    df_b[["Year", "Month"]]
)

train_mask_b = row_keys_b.isin(train_keys_b)
test_mask_b = row_keys_b.isin(test_keys_b)

X_train_b = X_b.loc[train_mask_b].reset_index(drop=True)
X_test_b = X_b.loc[test_mask_b].reset_index(drop=True)

y_train_b = y_b.loc[train_mask_b].reset_index(drop=True)
y_test_b = y_b.loc[test_mask_b].reset_index(drop=True)

print(f"Training rows: {len(X_train_b)}")
print(f"Testing rows: {len(X_test_b)}")

print(
    f"Training period: "
    f"{train_periods_b.iloc[0]['Year']}-"
    f"{train_periods_b.iloc[0]['Month']:02d} to "
    f"{train_periods_b.iloc[-1]['Year']}-"
    f"{train_periods_b.iloc[-1]['Month']:02d}"
)

print(
    f"Testing period: "
    f"{test_periods_b.iloc[0]['Year']}-"
    f"{test_periods_b.iloc[0]['Month']:02d} to "
    f"{test_periods_b.iloc[-1]['Year']}-"
    f"{test_periods_b.iloc[-1]['Month']:02d}"
)

# --------------------------------------------------------------------------
# TIME SERIES CROSS-VALIDATION for TASK 2
# --------------------------------------------------------------------------

row_periods_train_b = pd.MultiIndex.from_frame(
    df_b.loc[train_mask_b, ["Year", "Month"]].reset_index(drop=True)
)

cv_periods_b = train_periods_b.reset_index(drop=True)

tscv_b = TimeSeriesSplit(n_splits=3)

cv_b = []

for train_idx, val_idx in tscv_b.split(cv_periods_b):

    train_keys = pd.MultiIndex.from_frame(
        cv_periods_b.iloc[train_idx]
    )

    val_keys = pd.MultiIndex.from_frame(
        cv_periods_b.iloc[val_idx]
    )

    train_rows = np.flatnonzero(
        row_periods_train_b.isin(train_keys)
    )

    val_rows = np.flatnonzero(
        row_periods_train_b.isin(val_keys)
    )

    cv_b.append((train_rows, val_rows))

print(f"Task 2 TimeSeriesSplit folds: {len(cv_b)}")
print("Using RandomizedSearchCV (n_iter=12, cv=TimeSeriesSplit).\n")

# --------------------------------------------------------------------------
# MODEL DEFINITIONS for TASK 2
# --------------------------------------------------------------------------

baseline_models_b = {
    "Logistic Regression": make_model(
        LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            random_state=RANDOM_STATE
        )
    ),

    "Random Forest": make_model(
        RandomForestClassifier(
            class_weight="balanced",
            n_estimators=300,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    ),

    "SVM (RBF Kernel)": make_model(
        SVC(
            class_weight="balanced",
            kernel="rbf",
            C=1.0,
            gamma="scale",
            probability=True,
            random_state=RANDOM_STATE
        )
    )
}

search_configs_b = {
    "Logistic Regression": (
        make_model(
            LogisticRegression(
                class_weight="balanced",
                max_iter=2000,
                random_state=RANDOM_STATE
            )
        ), LR_GRID
    ),

    "Random Forest": (
        make_model(
            RandomForestClassifier(
                class_weight="balanced",
                random_state=RANDOM_STATE,
                n_jobs=-1
            )
        ), RF_GRID
    ),

    "SVM (RBF Kernel)": (
        make_model(
            SVC(
                class_weight="balanced",
                probability=True,
                random_state=RANDOM_STATE
            )
        ), SVM_GRID
    )
}

# --------------------------------------------------------------------------
# HYPERPARAMETER TUNING for TASK 2
# --------------------------------------------------------------------------

best_params_b = {}
tuned_models_b = {}

for name, (estimator, grid) in search_configs_b.items():

    t0 = time.time()

    search = RandomizedSearchCV(
        estimator,
        grid,
        n_iter=12,
        cv=cv_b,
        scoring="average_precision",
        random_state=RANDOM_STATE,
        n_jobs=1
    )

    search.fit(X_train_b, y_train_b)

    elapsed = time.time() - t0

    best_params_b[name] = search.best_params_
    tuned_models_b[name] = search.best_estimator_

    print(
        f"-> {name}: best temporal CV average_precision = "
        f"{search.best_score_:.4f} | "
        f"best params = {search.best_params_} | "
        f"{elapsed:.1f}s"
    )

rows = [
    {"Model": name, "Best Hyperparameters": str(params)}
    for name, params in best_params_b.items()
]

pd.DataFrame(rows).to_csv(
    "Extreme Best Hyperparameters.csv",
    index=False
)

# --------------------------------------------------------------------------
# BASELINE VS TUNED PERFORMANCE on TASK 2
# --------------------------------------------------------------------------

perf_rows = []

tuned_preds_b = {}
tuned_probs_b = {}

for name in baseline_models_b:

    baseline_models_b[name].fit(X_train_b, y_train_b)

    y_pred_base = baseline_models_b[name].predict(X_test_b)

    y_prob_base = baseline_models_b[name].predict_proba(
        X_test_b
    )[:, 1]

    y_pred_tuned = tuned_models_b[name].predict(X_test_b)

    y_prob_tuned = tuned_models_b[name].predict_proba(
        X_test_b
    )[:, 1]

    tuned_preds_b[name] = y_pred_tuned
    tuned_probs_b[name] = y_prob_tuned

    for label, y_pred, y_prob in [
        ("Baseline", y_pred_base, y_prob_base),
        ("Tuned", y_pred_tuned, y_prob_tuned)
    ]:

        perf_rows.append({
            "Model": name,
            "Version": label,

            "Precision (Extreme)": round(
                precision_score(y_test_b, y_pred, zero_division=0), 4
            ),

            "Recall (Extreme)": round(
                recall_score(y_test_b, y_pred, zero_division=0), 4
            ),

            "F1-Score (Extreme)": round(
                f1_score(y_test_b, y_pred, zero_division=0), 4
            ),

            "ROC-AUC": round(
                roc_auc_score(y_test_b, y_prob), 4
            ),

            "Average Precision (PR-AUC)": round(
                average_precision_score(y_test_b, y_prob), 4
            )
        })

extreme_comparison = pd.DataFrame(perf_rows)

print("\n" + extreme_comparison.to_string(index=False))

extreme_comparison.to_csv(
    "Extreme_tuned_vs_baseline.csv",
    index=False
)

# --------------------------------------------------------------------------
# EXTREME RAINFALL PERFORMANCE PLOT - Task 2
# --------------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(9, 5.5))

pivot = extreme_comparison.pivot(
    index="Model",
    columns="Version",
    values="Average Precision (PR-AUC)"
)

pivot[["Baseline", "Tuned"]].plot(
    kind="bar",
    ax=ax,
    color=["#B0BEC5", "#D64545"]
)

ax.set_title(
    "Task 2: Extreme Rainfall — Baseline vs Tuned — Average Precision"
)

ax.set_ylabel("Average Precision (PR-AUC)")

plt.xticks(rotation=0)
plt.tight_layout()

plt.savefig("Extreme--Tuned vs Baseline.png")
plt.close()

print("\nSaved: Extreme--Tuned vs Baseline.png")

# --------------------------------------------------------------------------
# CONFUSION MATRICES - EXTREME RAINFALL Modelling
# --------------------------------------------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

for ax, (name, y_pred) in zip(axes, tuned_preds_b.items()):

    cm = confusion_matrix(
        y_test_b,
        y_pred,
        labels=[0, 1]
    )

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Not Extreme", "Extreme"]
    )

    disp.plot(
        ax=ax,
        cmap="Reds",
        colorbar=False
    )

    ax.set_title(f"{name} (Tuned)")
    ax.set_xlabel("Forecasted Class")
    ax.set_ylabel("Actual Class")

fig.suptitle(
    "Confusion Matrices: Extreme Rainfall Forecast — Tuned Models",
    fontweight="bold",
    fontsize=14
)

plt.tight_layout()

plt.savefig(
    "Confusion Matrices Task 2: Extreme Rainfall Forecast--Tuned Models.png"
)

plt.close()

print(
    "Saved: Confusion Matrices Task 2: Extreme Rainfall Forecast--Tuned Models.png"
)

# ==========================================================================
# COMPLETION
# ==========================================================================

print("\n" + "=" * 140)
print("CHRONOLOGICAL MODEL TRAINING AND TUNING FULLY COMPLETE")
print("=" * 140)


TASK 1: SEASONAL RAINFALL -- CHRONOLOGICAL
Training rows: 637
Testing rows: 161
Training period: 1984 - 2014
Testing period: 2014 - 2021
Task 1 TimeSeriesSplit folds: 5
-> Logistic Regression: best temporal CV f1_macro = 0.2546 | best params = {'model__C': 10} | 0.2s
-> Random Forest: best temporal CV f1_macro = 0.2260 | best params = {'model__max_depth': 10, 'model__min_samples_split': 2, 'model__n_estimators': 100} | 14.3s
-> SVM (RBF Kernel): best temporal CV f1_macro = 0.2860 | best params = {'model__C': 100, 'model__gamma': 0.1, 'model__kernel': 'rbf'} | 0.6s

              Model  Version  Accuracy  F1-Score (macro)
Logistic Regression Baseline    0.2112            0.1856
Logistic Regression    Tuned    0.2112            0.1804
      Random Forest Baseline    0.3230            0.3092
      Random Forest    Tuned    0.2733            0.2644
   SVM (RBF Kernel) Baseline    0.1988            0.1721
   SVM (RBF Kernel)    Tuned    0.3292            0.2689

Saved: Seasonal Tuned vs Bas